<a href="https://colab.research.google.com/github/iras-mpark/MLA1020/blob/main/week11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [38]:
from dataclasses import dataclass
from typing import Any

from typing import Callable
Policy = Callable[[Any], Any]

from functools import partial


import numpy as np

from collections import defaultdict

@dataclass(frozen=True)
class Step:
    action: Any
    prob: float  # New: the probability that we ended up here
    reward: float
    state: Any
class MDP:
    def start_state(self) -> Any:
        raise NotImplementedError

    def successors(self, state: Any) -> list[Step]:
        raise NotImplementedError

    def is_end(self, state: Any) -> bool:
        raise NotImplementedError
    def discount(self) -> float:
        raise NotImplementedError
class FlakyTramMDP(MDP):
    def __init__(self, num_locs: int, failure_prob: float):
        self.num_locs = num_locs
        self.failure_prob = failure_prob
    def start_state(self) -> Any:
        return 1

    def successors(self, state: Any) -> list[Step]:
        successors = []
        # Walk
        if state + 1 <= self.num_locs:
            successors.append(Step(action="walk", prob=1, reward=-1, state=state + 1))
        # Tram
        if 2 * state <= self.num_locs:
            # Success: move to desired state
            successors.append(Step(action="tram", prob=1 - self.failure_prob, reward=-2, state=2 * state))
            # Failure: stay in the same state
            successors.append(Step(action="tram", prob=self.failure_prob, reward=-2, state=state))
        return successors
    def is_end(self, state: Any) -> bool:
        return state == self.num_locs
    def discount(self) -> float:
        # No discounting for now
        return 1

mdp = FlakyTramMDP(num_locs=10, failure_prob=0.4)
np.random.seed(1)

class RLAlgorithm:
    """
    Abstract class for an RL algorithm, which does two things:
    1. Sends actions to the environment
    2. Incorporates feedback from the environment
    """
    def get_action(self, state: Any) -> Any:
        raise NotImplementedError
    def incorporate_feedback(self, state: Any, action: Any, reward: Any, next_state: Any, is_end: bool) -> None:
        raise NotImplementedError

class StaticAgent(RLAlgorithm):
    def __init__(self, policy: Policy):
        self.policy = policy
    def get_action(self, state: Any) -> Any:
        return self.policy(state)
    def incorporate_feedback(self, state: Any, action: Any, reward: Any, next_state: Any, is_end: bool) -> None:
        # Setup Reward Feedback Loigic
        pass

def sample_transition(mdp: MDP, state: Any, action: Any) -> Step:
    """
    Samples a transition from the MDP: samples s' with probability T(s, a, s').
    Returns:
    - reward: the reward for the transition
    - next_state: the next state
    - is_end: whether the next state is an end state
    """
    # Get successors given (state, action)
    successors = [successor for successor in mdp.successors(state) if successor.action == action]
    if len(successors) == 0:
        raise ValueError(f"No successors found for state {state} and action {action}")
    # Sample a successor based on its probabilities
    probs = [successor.prob for successor in successors]
    choice = np.random.choice(len(successors), p=probs)
    step = successors[choice]
    return step

def compute_utility(steps: list[Step], discount: float) -> float:
    """Computes the utility (discounted sum of rewards) of a rollout."""
    rewards = [step.reward * discount ** i for i, step in enumerate(steps)]
    utility = sum(rewards)
    return utility

class Rollout:
    """Represents a rollout of an MDP (sequence of actions that produces a utility)."""
    steps: list[Step]
    discount: float
    utility: float  # Discounted sum of rewards
    def __init__(self, steps: list[Step], discount: float):
        self.steps = steps
        self.discount = discount
        self.utility = compute_utility(steps, discount)

def simulate(mdp: MDP, rl: RLAlgorithm, num_trials: int) -> float:
    """
    Runs the RL algorithm on the MDP.
    Return the mean utility of the rollouts.
    """
    utilities = []
    # Repeat multiple times
    for trial in range(num_trials):
        # Environment state
        state = mdp.start_state()
        steps = []
        while not mdp.is_end(state):
            # Agent sends action to environment
            action = rl.get_action(state)
            # Environment sends reward and next state to agent
            step = sample_transition(mdp, state, action)
            rl.incorporate_feedback(state, action, step.reward, step.state, mdp.is_end(step.state))
            steps.append(step)
            state = step.state
        # Compute utility of this rollout
        rollout = Rollout(steps=steps, discount=mdp.discount())
        utilities.append(rollout.utility)
    mean_utility = np.mean(utilities).item()
    return mean_utility

LEADERBOARD = {}  # method -> value

def update_leaderboard(method: str, value: float) -> dict[str, float]:
    LEADERBOARD[method] = value
    return LEADERBOARD

# Model-free RL

In [39]:
# Tram 탈 수 있으면 확률적으로 tram을 타고, 그 외에는 walk를 항상 선택하는 policy

def walk_tram_policy(num_locs: int, state: Any) -> Any:
    """Chooses a random valid action."""
    if 2 * state <= num_locs:
        # Can take the tram
        return np.random.choice(["walk", "tram"]).item()
    else:
        # Can only walk
        return "walk"

In [40]:
def try_out_exploration_policy(exploration_policy: Policy):
    state=1
    action = exploration_policy(state)
    print(f"Action taken: {action}, at State {state}")
    action = exploration_policy(state)
    print(f"Action taken: {action}, at State {state}")
    action = exploration_policy(state)
    print(f"Action taken: {action}, at State {state}")

    state=6
    action = exploration_policy(state=6)
    print(f"Action taken: {action}, at State {state}")


In [41]:
exploration_policy = partial(walk_tram_policy, mdp.num_locs)
try_out_exploration_policy(exploration_policy)

Action taken: tram, at State 1
Action taken: tram, at State 1
Action taken: walk, at State 1
Action taken: walk, at State 6


#### SARSA

In [42]:
class SARSA(RLAlgorithm):
    def __init__(
        self,
        exploration_policy: Policy,
        epsilon: float,
        discount: float,
        learning_rate: float
    ):
        self.exploration_policy = exploration_policy
        self.epsilon = epsilon
        self.discount = discount
        self.learning_rate = learning_rate

        # state -> action -> Q-value
        self.Q = defaultdict(lambda: defaultdict(float))

    def get_action(self, state: Any) -> Any:

        # No prior knowledge for this state
        if len(self.Q[state]) == 0:
            action = self.exploration_policy(state)

            print(
                f"State {state}: Choosing random action {action} "
                f"(initial exploration - no known Q-values)"
            )

            return action

        # Epsilon-greedy exploration
        if np.random.random() < self.epsilon:

            action = self.exploration_policy(state)

            print(
                f"State {state}: Choosing random action {action} "
                f"(epsilon-greedy exploration, epsilon={self.epsilon})"
            )

            return action

        # Exploitation
        else:

            action = self.pi(state)

            print(
                f"State {state}: Choosing best action {action} "
                f"(exploitation from current Q-values)"
            )

            return action

    def pi(self, state: Any) -> Any:
        """Return the greedy policy corresponding to current Q-values."""

        actions = list(self.Q[state].keys())

        if len(actions) == 0:
            print(f"State {state}: No available actions in Q-table yet.")
            return None

        q_values = [self.Q[state][action] for action in actions]

        print(f"\n[Policy Evaluation for State {state}]")
        for action, q_value in zip(actions, q_values):
            print(f"  Q({state}, {action}) = {q_value:.4f}")

        best_action = actions[np.argmax(q_values).item()]

        print(f"  -> Best action selected: {best_action}")

        return best_action

    def incorporate_feedback(
        self,
        state: Any,
        action: Any,
        reward: Any,
        next_state: Any,
        is_end: bool
    ) -> None:

        print("\n" + "=" * 70)
        print("[SARSA Feedback Incorporation]")
        print(
            f"Observed transition:\n"
            f"  state={state}\n"
            f"  action={action}\n"
            f"  reward={reward}\n"
            f"  next_state={next_state}\n"
            f"  is_end={is_end}"
        )

        # Current Q before update
        old_q = self.Q[state][action]

        print(f"\nCurrent Q-value:")
        print(f"  Q({state}, {action}) = {old_q:.4f}")

        # Terminal state case
        if is_end:

            next_action = None
            next_q = 0.0

            print("\nTerminal state reached.")
            print("  No next action.")
            print("  Future utility = 0")

        else:

            # SARSA uses ON-POLICY next action
            next_action = self.get_action(next_state)

            next_q = self.Q[next_state].get(next_action, 0.0)

            print("\n[On-Policy Next Step]")
            print(f"  Next action chosen by current policy: {next_action}")
            print(f"  Q({next_state}, {next_action}) = {next_q:.4f}")

        # SARSA target:
        # target = r + gamma * Q(s', a')
        utility = reward + self.discount * next_q

        print("\n[SARSA Target Calculation]")
        print(
            f"  utility = reward + discount * next_Q\n"
            f"          = {reward} + ({self.discount} * {next_q:.4f})\n"
            f"          = {utility:.4f}"
        )

        # TD Error
        td_error = utility - old_q

        print("\n[TD Error]")
        print(
            f"  TD Error = target - current_Q\n"
            f"           = {utility:.4f} - {old_q:.4f}\n"
            f"           = {td_error:.4f}"
        )

        # Update rule:
        # Q(s,a) <- Q(s,a) + alpha * TD_error
        update_amount = self.learning_rate * td_error

        self.Q[state][action] += update_amount

        new_q = self.Q[state][action]

        print("\n[Q-value Update]")
        print(
            f"  Q({state}, {action}) ← {old_q:.4f} + "
            f"{self.learning_rate} * ({td_error:.4f})"
        )

        print(f"  Update amount = {update_amount:.4f}")
        print(f"  New Q({state}, {action}) = {new_q:.4f}")

        # Full Q-table snapshot
        print("\n[Updated Q-table Snapshot]")
        for s in self.Q.keys():
            for a in self.Q[s].keys():
                print(f"  Q({s}, {a}) = {self.Q[s][a]:.4f}")

        print("=" * 70)

In [43]:
exploration_policy = partial(walk_tram_policy, mdp.num_locs)
rl = SARSA(exploration_policy=exploration_policy, epsilon=0.4, discount=1, learning_rate=0.1)

In [44]:
value = simulate(mdp, rl, num_trials=20)
leaderboard = update_leaderboard("sarsa", value)

State 1: Choosing random action walk (initial exploration - no known Q-values)

[SARSA Feedback Incorporation]
Observed transition:
  state=1
  action=walk
  reward=-1
  next_state=2
  is_end=False

Current Q-value:
  Q(1, walk) = 0.0000
State 2: Choosing random action tram (initial exploration - no known Q-values)

[On-Policy Next Step]
  Next action chosen by current policy: tram
  Q(2, tram) = 0.0000

[SARSA Target Calculation]
  utility = reward + discount * next_Q
          = -1 + (1 * 0.0000)
          = -1.0000

[TD Error]
  TD Error = target - current_Q
           = -1.0000 - 0.0000
           = -1.0000

[Q-value Update]
  Q(1, walk) ← 0.0000 + 0.1 * (-1.0000)
  Update amount = -0.1000
  New Q(1, walk) = -0.1000

[Updated Q-table Snapshot]
  Q(1, walk) = -0.1000
State 2: Choosing random action tram (initial exploration - no known Q-values)

[SARSA Feedback Incorporation]
Observed transition:
  state=2
  action=tram
  reward=-2
  next_state=4
  is_end=False

Current Q-value:
  Q

#### Q-Learning

In [45]:
class QLearning(SARSA):
    """Q-learning is SARSA, but with an off-policy greedy target policy."""

    def incorporate_feedback(
        self,
        state: Any,
        action: Any,
        reward: Any,
        next_state: Any,
        is_end: bool
    ) -> None:

        print("\n" + "=" * 70)
        print("[Q-Learning Feedback Incorporation]")
        print(
            f"Observed transition:\n"
            f"  state={state}\n"
            f"  action={action}\n"
            f"  reward={reward}\n"
            f"  next_state={next_state}\n"
            f"  is_end={is_end}"
        )

        # Current Q before update
        old_q = self.Q[state][action]

        print(f"\nCurrent Q-value:")
        print(f"  Q({state}, {action}) = {old_q:.4f}")

        # Terminal state
        if is_end:

            next_action = None
            next_q = 0.0

            print("\nTerminal state reached.")
            print("  No future action needed.")
            print("  Future utility = 0")

        else:

            print("\n[Off-Policy Greedy Evaluation]")
            print(
                "  Q-Learning ignores exploration policy here.\n"
                "  It evaluates the GREEDY best next action using pi(next_state)."
            )

            # OFF-POLICY:
            # Use greedy policy, not exploratory get_action
            next_action = self.pi(next_state)

            if next_action is None:
                next_q = 0.0

                print(
                    f"  No known actions yet for next_state={next_state}.\n"
                    f"  Default next Q = 0"
                )

            else:
                next_q = self.Q[next_state].get(next_action, 0.0)

                print(f"  Best greedy next action: {next_action}")
                print(f"  Q({next_state}, {next_action}) = {next_q:.4f}")

        # Q-Learning target:
        # target = r + gamma * max_a' Q(s', a')
        utility = reward + self.discount * next_q

        print("\n[Q-Learning Target Calculation]")
        print(
            f"  utility = reward + discount * max_next_Q\n"
            f"          = {reward} + ({self.discount} * {next_q:.4f})\n"
            f"          = {utility:.4f}"
        )

        # TD Error
        td_error = utility - old_q

        print("\n[TD Error]")
        print(
            f"  TD Error = target - current_Q\n"
            f"           = {utility:.4f} - {old_q:.4f}\n"
            f"           = {td_error:.4f}"
        )

        # Update:
        # Q(s,a) <- Q(s,a) + alpha * TD_error
        update_amount = self.learning_rate * td_error

        self.Q[state][action] += update_amount

        new_q = self.Q[state][action]

        print("\n[Q-value Update]")
        print(
            f"  Q({state}, {action}) ← {old_q:.4f} + "
            f"{self.learning_rate} * ({td_error:.4f})"
        )

        print(f"  Update amount = {update_amount:.4f}")
        print(f"  New Q({state}, {action}) = {new_q:.4f}")

        # Compare SARSA vs Q-Learning conceptually
        print("\n[Key Q-Learning Characteristic]")
        print(
            "  Target policy: GREEDY (max Q)\n"
            "  Behavior policy: May still explore externally\n"
            "  => OFF-POLICY learning"
        )

        # Full Q-table snapshot
        print("\n[Updated Q-table Snapshot]")
        for s in self.Q.keys():
            for a in self.Q[s].keys():
                print(f"  Q({s}, {a}) = {self.Q[s][a]:.4f}")

        print("=" * 70)

In [46]:
rl = QLearning(exploration_policy=exploration_policy, epsilon=0.4, discount=1, learning_rate=0.1)


In [47]:
value = simulate(mdp, rl, num_trials=10)
leaderboard = update_leaderboard("q_learning", value)

State 1: Choosing random action walk (initial exploration - no known Q-values)

[Q-Learning Feedback Incorporation]
Observed transition:
  state=1
  action=walk
  reward=-1
  next_state=2
  is_end=False

Current Q-value:
  Q(1, walk) = 0.0000

[Off-Policy Greedy Evaluation]
  Q-Learning ignores exploration policy here.
  It evaluates the GREEDY best next action using pi(next_state).
State 2: No available actions in Q-table yet.
  No known actions yet for next_state=2.
  Default next Q = 0

[Q-Learning Target Calculation]
  utility = reward + discount * max_next_Q
          = -1 + (1 * 0.0000)
          = -1.0000

[TD Error]
  TD Error = target - current_Q
           = -1.0000 - 0.0000
           = -1.0000

[Q-value Update]
  Q(1, walk) ← 0.0000 + 0.1 * (-1.0000)
  Update amount = -0.1000
  New Q(1, walk) = -0.1000

[Key Q-Learning Characteristic]
  Target policy: GREEDY (max Q)
  Behavior policy: May still explore externally
  => OFF-POLICY learning

[Updated Q-table Snapshot]
  Q(1, w